[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/rag-pipeline-practice/01_web_crawling/01_web_crawling_solutions.ipynb)

# 01. 웹 크롤링 실습 — 연습 문제 해설

[01_web_crawling.ipynb](01_web_crawling.ipynb) 끝의 연습 문제 2개에 대한 정답 코드와 해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q requests beautifulsoup4 python-dotenv

In [ ]:
import time
import sqlite3
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; tutorial-crawler/1.0)"}


def fetch(url: str) -> requests.Response:
    response = requests.get(url, headers=HEADERS, timeout=10)
    response.raise_for_status()
    if response.encoding is None or response.encoding.lower() == "iso-8859-1":
        response.encoding = response.apparent_encoding
    return response


def extract_text_from_html(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator="\n")
    lines = [line.strip() for line in text.splitlines()]
    return "\n".join(line for line in lines if line)

## 연습 1. 태그 페이지 여러 개 순회하며 명언 개수 세기

In [ ]:
TAG_TARGETS = [
    "https://quotes.toscrape.com/tag/love/",
    "https://quotes.toscrape.com/tag/inspirational/",
    "https://quotes.toscrape.com/tag/life/",
]

results = {}
for url in TAG_TARGETS:
    page = fetch(url)
    soup = BeautifulSoup(page.text, "html.parser")
    quote_count = len(soup.select("div.quote"))
    results[url] = quote_count
    print(f"{url} -> {quote_count}개")
    time.sleep(0.5)

print("\n총합:", sum(results.values()))

**해설**
- `div.quote` CSS 선택자로 한 페이지 안의 명언 블록 개수를 셉니다. 태그 페이지마다 담긴 명언 수가 다르기 때문에 값이 다르게 나옵니다.
- 페이지마다 `time.sleep()`으로 딜레이를 주는 것은 크롤링 대상 서버에 부담을 주지 않기 위한 예의이자, 실전에서는 IP 차단을 피하기 위한 안전장치이기도 합니다.

## 연습 2. UPSERT로 중복 없이 갱신되는지 확인

In [ ]:
CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS crawled_documents (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    url TEXT NOT NULL UNIQUE,
    content_type TEXT NOT NULL,
    text_content TEXT,
    crawled_at TEXT NOT NULL DEFAULT (datetime('now'))
);
"""

UPSERT_SQL = """
INSERT INTO crawled_documents (url, content_type, text_content, crawled_at)
VALUES (?, ?, ?, datetime('now'))
ON CONFLICT(url) DO UPDATE SET
    content_type = excluded.content_type,
    text_content = excluded.text_content,
    crawled_at = datetime('now');
"""

DB_PATH = "crawled_check.db"
with sqlite3.connect(DB_PATH) as conn:
    conn.execute(CREATE_TABLE_SQL)

url = "https://quotes.toscrape.com/"

def crawl_and_save():
    page = fetch(url)
    text = extract_text_from_html(page.text)
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(UPSERT_SQL, (url, "html", text))


crawl_and_save()
time.sleep(1)  # crawled_at 값이 눈에 띄게 달라지도록 잠깐 대기
crawl_and_save()

with sqlite3.connect(DB_PATH) as conn:
    rows = conn.execute("SELECT id, url, crawled_at FROM crawled_documents WHERE url = ?", (url,)).fetchall()

print("행 개수 (1개여야 정상):", len(rows))
for row in rows:
    print(row)

**해설**
- 같은 URL로 두 번 저장해도 행이 1개만 남습니다 — `ON CONFLICT(url) DO UPDATE`가 "이미 있는 url이면 새로 넣지 말고 내용만 갱신하라"고 지시하기 때문입니다.
- 만약 `ON CONFLICT` 절 없이 단순 `INSERT`만 썼다면 `url TEXT NOT NULL UNIQUE` 제약 때문에 두 번째 실행에서 에러가 나거나(제약 위반), UNIQUE 제약이 없었다면 중복 행이 계속 쌓였을 것입니다.
- `crawled_at`이 두 번째 실행 시각으로 갱신된 것도 확인할 수 있습니다 — "같은 페이지를 다시 크롤링하면 최신 정보로 덮어쓴다"는 원본 프로젝트의 의도가 그대로 재현된 것입니다.